In [1]:
import json

In [2]:
with open("../../data/retacred/test.json") as file:
  data = json.load(file)

In [3]:
# get the sequence lengths
seq_lens = []
for d in data:
  seq_lens.append((d['id'], len(d['token'])))
  
seq_lens = sorted(list(set(seq_lens)), key=lambda x: x[1], reverse=False)
print(len(seq_lens))
print(seq_lens[0])

13418
('098f6eb6b01369386139', 4)


In [4]:
num_docs = len(seq_lens)

bottom_perc = 0.15
top_perc = 0.85

bottom_ids = []
mid_ids = []
top_ids = []
# get the IDs
for i, (did, dtok) in enumerate(seq_lens):
    if i / num_docs <= bottom_perc:
        bottom_ids.append(did)
    elif i / num_docs >= top_perc:
        top_ids.append(did)
    else:
        mid_ids.append(did)

print(len(bottom_ids)/ num_docs)
print(len(mid_ids)/ num_docs)
print(len(top_ids)/ num_docs)


0.15002235802653152
0.7000298107020421
0.14994783127142644


In [5]:
bottom_test = []
mid_test = []
top_test = []

for d in data:
  if d['id'] in bottom_ids:
      bottom_test.append(d)
  elif d['id'] in mid_ids:
      mid_test.append(d)
  elif d['id'] in top_ids:
      top_test.append(d)
  else:
      print("error")
      
print(len(bottom_test))
print(len(mid_test))
print(len(top_test))

2013
9393
2012


In [6]:
# write new json files for top, mid, and bottom test
with open("../../data/retacred/test_bottom.json", "w") as file:
  json.dump(bottom_test, file)
with open("../../data/retacred/test_mid.json", "w") as file:
  json.dump(mid_test, file)
with open("../../data/retacred/test_top.json", "w") as file:
  json.dump(top_test, file)

In [7]:
# CD into the parent folder for convinience

%cd ../../src_ra_cgcn
%pwd

c:\Users\Alex\Desktop\Text Mining Coursework\COMP61332-re\src_ra_cgcn


'c:\\Users\\Alex\\Desktop\\Text Mining Coursework\\COMP61332-re\\src_ra_cgcn'

In [8]:
# imports

import random
import argparse

from tqdm import tqdm
import torch

from data.loader import DataLoader, DataLoaderPredict
from model.trainer import GCNTrainer
from utils import torch_utils, scorer, constant, helper
from utils.vocab import Vocab


In [9]:
def load_model(model_dir, model="best_model.pt", seed=1234, cuda=None):
    # set cuda and cuda seed
    if cuda is None:
        cuda = torch.cuda.is_available()
        torch.cuda.manual_seed(seed)

    # set the seeds
    torch.manual_seed(seed)
    random.seed(seed)

    # load opt
    model_file = f"{model_dir}/{model}"
    print(f"Loading model from {model_file}")
    opt = torch_utils.load_config(model_file)
    trainer = GCNTrainer(opt)
    trainer.load(model_file)

    # load vocab
    vocab_file = f"{model_dir}/vocab.pkl"
    vocab = Vocab(vocab_file, load=True)
    assert opt['vocab_size'] == vocab.size, "Vocab size must match that in the saved model."

    return trainer, vocab, opt

In [10]:
def evaluate_model(trainer, vocab, data_dir="../data/retacred", opt=None, scorer=None, test_file="test.json"):
    assert opt is not None, "opt must not be None."
    assert scorer is not None, "scorer must not be None."

    # load data
    data_file = f"{data_dir}/{test_file}"
    print(f"Loading data from {data_file} with batch size {opt['batch_size']}...")
    batch = DataLoader(data_file, opt['batch_size'], opt, vocab, evaluation=True)

    label2id = constant.LABEL_TO_ID
    id2label = dict([(v,k) for k,v in label2id.items()])

    predictions = []
    all_probs = []
    batch_iter = tqdm(batch)
    for i, b in enumerate(batch_iter):
        preds, probs, _ = trainer.predict(b)
        predictions += preds
        all_probs += probs

    predictions = [id2label[p] for p in predictions]
    _, details = scorer.score(batch.gold(), predictions, verbose=False)

    # print("Detailed evaluation:")
    # print("\n".join([f"{k}: {v}" for k, v in details.items()]))
    print("Evaluation ended.")

In [11]:
trainer_agcn, vocab_agcn, opt_agcn = load_model(model_dir="saved_models/400/", model="best_model.pt")
trainer_gcn, vocab_gcn, opt_gcn = load_model(model_dir="saved_models/500/", model="best_model.pt")


Loading model from saved_models/400//best_model.pt


c:\Users\Alex\Desktop\Text Mining Coursework\COMP61332-re\src_ra_cgcn\utils\torch_utils.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dump = torch.load(filename)


Finetune all embeddings.
Vocab size 50115 loaded from file
Loading model from saved_models/500//best_model.pt


c:\Users\Alex\Desktop\Text Mining Coursework\COMP61332-re\src_ra_cgcn\model\trainer.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename)


Finetune all embeddings.
Vocab size 50115 loaded from file


In [12]:
# eval for attention GCN (bottom)
evaluate_model(trainer=trainer_agcn, vocab=vocab_agcn, opt=opt_agcn, scorer=scorer, test_file="test_bottom.json")

Loading data from ../data/retacred/test_bottom.json with batch size 50...
41 batches created for ../data/retacred/test_bottom.json


100%|██████████| 41/41 [00:01<00:00, 34.98it/s]

Precision (micro): 77.189%
   Recall (micro): 81.070%
       F1 (micro): 79.082%
Evaluation ended.


In [13]:
# eval for attention GCN (mid)
evaluate_model(trainer=trainer_agcn, vocab=vocab_agcn, opt=opt_agcn, scorer=scorer, test_file="test_mid.json")

Loading data from ../data/retacred/test_mid.json with batch size 50...
188 batches created for ../data/retacred/test_mid.json


100%|██████████| 188/188 [00:03<00:00, 49.50it/s]

Precision (micro): 78.864%
   Recall (micro): 76.772%
       F1 (micro): 77.804%
Evaluation ended.


In [14]:
# eval for attention GCN (top)
evaluate_model(trainer=trainer_agcn, vocab=vocab_agcn, opt=opt_agcn, scorer=scorer, test_file="test_top.json")

Loading data from ../data/retacred/test_top.json with batch size 50...
41 batches created for ../data/retacred/test_top.json


100%|██████████| 41/41 [00:01<00:00, 32.77it/s]

Precision (micro): 76.887%
   Recall (micro): 72.018%
       F1 (micro): 74.373%
Evaluation ended.


In [15]:
# eval for GCN (bottom)
evaluate_model(trainer=trainer_gcn, vocab=vocab_gcn, opt=opt_gcn, scorer=scorer, test_file="test_bottom.json")

Loading data from ../data/retacred/test_bottom.json with batch size 50...
41 batches created for ../data/retacred/test_bottom.json


100%|██████████| 41/41 [00:00<00:00, 70.31it/s]


Precision (micro): 76.430%
   Recall (micro): 82.888%
       F1 (micro): 79.528%
Evaluation ended.


In [16]:
# eval for GCN (mid)
evaluate_model(trainer=trainer_gcn, vocab=vocab_gcn, opt=opt_gcn, scorer=scorer, test_file="test_mid.json")

Loading data from ../data/retacred/test_mid.json with batch size 50...
188 batches created for ../data/retacred/test_mid.json


100%|██████████| 188/188 [00:04<00:00, 38.87it/s]

Precision (micro): 75.146%
   Recall (micro): 76.450%
       F1 (micro): 75.793%
Evaluation ended.


In [17]:
# eval for GCN (top)
evaluate_model(trainer=trainer_gcn, vocab=vocab_gcn, opt=opt_gcn, scorer=scorer, test_file="test_top.json")

Loading data from ../data/retacred/test_top.json with batch size 50...
41 batches created for ../data/retacred/test_top.json


100%|██████████| 41/41 [00:01<00:00, 36.89it/s]

Precision (micro): 73.919%
   Recall (micro): 75.552%
       F1 (micro): 74.727%
Evaluation ended.
